In [1]:
import torch
import numpy as np
from PIL import Image
import t2v_metrics


# ---------------------------------------------------
# 1) Инициализация VLM / VQAScore модели один раз
# ---------------------------------------------------
def build_vlm_scorer(device: str = None, model_name: str = "clip-flant5-xl"):
    """
    Создает VQAScore-модель один раз, чтобы не грузить ее на каждый вызов.
    """
    if device is None:
        # device = "cuda" if torch.cuda.is_available() else "cpu"
        device = 'mps'

    scorer = t2v_metrics.VQAScore(model=model_name, device=device)
    return scorer


# ---------------------------------------------------
# 2) Конвертация tensor image -> PIL
# ---------------------------------------------------
def image_tensor_to_pil(image: torch.Tensor) -> Image.Image:
    """
    Принимает изображение в одном из форматов:
      - [3, H, W]
      - [1, 3, H, W]
    со значениями в [0, 1]
    и возвращает PIL.Image.
    """
    if image.ndim == 4:
        if image.shape[0] != 1:
            raise ValueError(f"Ожидался batch размером 1, получено: {image.shape}")
        image = image[0]

    if image.ndim != 3 or image.shape[0] != 3:
        raise ValueError(f"Ожидался tensor формата [3,H,W] или [1,3,H,W], получено: {image.shape}")

    image = image.detach().float().cpu().clamp(0, 1)
    image_np = (image.permute(1, 2, 0).numpy() * 255.0).round().astype(np.uint8)
    return Image.fromarray(image_np)


# ---------------------------------------------------
# 3) Score по уже декодированному изображению
# ---------------------------------------------------
@torch.no_grad()
def compute_vlm_score_from_image(
    image: torch.Tensor,
    prompt: str,
    vqa_scorer,
    question_template: str = None,
    answer_template: str = None,
):
    """
    Считает VLM/VQAScore между изображением и prompt.

    Параметры
    ---------
    image:
        torch.Tensor в формате [3,H,W] или [1,3,H,W], значения в [0,1]
    prompt:
        текстовый prompt
    vqa_scorer:
        объект t2v_metrics.VQAScore(...)
    question_template, answer_template:
        необязательно. Если не передавать, используются дефолты модели.

    Возвращает
    ----------
    float
        scalar score
    """
    pil_image = image_tensor_to_pil(image)

    kwargs = {}
    if question_template is not None:
        kwargs["question_template"] = question_template
    if answer_template is not None:
        kwargs["answer_template"] = answer_template

    scores = vqa_scorer(
        images=[pil_image],
        texts=[prompt],
        **kwargs,
    )

    # Обычно возвращается tensor формы [1,1] либо совместимый scalar-like объект
    if isinstance(scores, torch.Tensor):
        return float(scores.squeeze().detach().cpu().item())

    return float(np.array(scores).squeeze())


# ---------------------------------------------------
# 4) Score напрямую из латентов
# ---------------------------------------------------
@torch.no_grad()
def compute_vlm_score_from_latents(
    latents: torch.Tensor,
    prompt: str,
    vae,
    vqa_scorer,
):
    """
    Декодирует латенты через VAE и считает VLM/VQAScore.
    latents: [1,4,H/8,W/8]
    """
    image = decode_latents(latents)   # твоя функция из пайплайна, вернет [1,3,H,W] в [0,1]
    score = compute_vlm_score_from_image(
        image=image,
        prompt=prompt,
        vqa_scorer=vqa_scorer,
    )
    return score


# ---------------------------------------------------
# 5) Удобная версия: score для всех промежуточных шагов
# ---------------------------------------------------
@torch.no_grad()
def compute_vlm_scores_for_saved_steps(
    latents_history: list[torch.Tensor],
    prompt: str,
    vae,
    vqa_scorer,
):
    """
    latents_history: список латентов по шагам denoising.
    Возвращает список float-скоров той же длины.
    """
    scores = []
    for step_latents in latents_history:
        score = compute_vlm_score_from_latents(
            latents=step_latents,
            prompt=prompt,
            vae=vae,
            vqa_scorer=vqa_scorer,
        )
        scores.append(score)
    return scores


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/homebrew/Cellar/python@3.10/3.10.16/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/homebrew/Cellar/python@3.10/3.10.16/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/Users/alexkarachun/Documents/DEV/course-work-diffusions/venv/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_i

AttributeError: module 'torch' has no attribute 'xpu'

In [ ]:
print(1)

In [2]:
pip install -U torch

  Using cached torch-2.10.0-2-cp310-none-macosx_11_0_arm64.whl.metadata (31 kB)
Using cached torch-2.10.0-2-cp310-none-macosx_11_0_arm64.whl (79.4 MB)
  Attempting uninstall: torch
    Found existing installation: torch 2.1.0
    Uninstalling torch-2.1.0:
      Successfully uninstalled torch-2.1.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.16.0 requires torch==2.1.0, but you have torch 2.10.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import torch

print(torch.xpu.is_available())

AttributeError: module 'torch' has no attribute 'xpu'